# 《Excel数据可视化——从图表到数据大屏》第三章「动态图表」Python 迁移实现

> **实验课程**：《大数据分析及数据可视化》
> **实验内容**：将 Excel 第三章 7 个动态图表案例迁移为 Python 实现，并组合成数据大屏
> **技术栈**：pandas（数据处理）+ pyecharts（交互式可视化，底层为 Apache ECharts）

## Excel 动态机制 → Python 对应实现

| Excel 中的动态机制 | Python / pyecharts 对应组件 |
|---|---|---|
| 控件单元格 + INDEX/VLOOKUP 取数 | `Timeline` 时间轴组件（拖动/点击切换） |
| 复选框 + IF 控制系列显隐 | 图例 `Legend` 点击切换系列显隐 |
| 数据透视表 + 切片器 | `pandas.pivot_table / groupby` + `Tab` 选项卡 |
| VBA 宏按钮（近7天/近30天） | `Tab` 选项卡按钮 |
| 组合框 + 选项按钮 | `Timeline` 时间轴组件 |

---

In [ ]:
# ============================================================
# 0. 环境准备与数据加载
# ============================================================
from pathlib import Path
import pandas as pd
from pyecharts import options as opts
from pyecharts.commons.utils import JsCode
from pyecharts.charts import Bar, Pie, Line, Timeline, Tab, EffectScatter, Page
import pyecharts

print("pyecharts 版本:", pyecharts.__version__)
print("pandas 版本:", pd.__version__)

DATA_DIR = Path("data")
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

# 统一配色（与 Excel 默认主题接近的商务配色）
PALETTE = ["#4472C4", "#ED7D31", "#A5A5A5", "#FFC000", "#5B9BD5", "#70AD47"]

# 读取全部案例数据（由《第三章 动态图表.xlsm》1:1 提取）
df1 = pd.read_csv(DATA_DIR / "案例1_区域月度销售额.csv")
df2 = pd.read_csv(DATA_DIR / "案例2_部门月度人数.csv")
df3 = pd.read_csv(DATA_DIR / "案例3_流量来源占比.csv")
df4 = pd.read_csv(DATA_DIR / "案例4_化妆品销售.csv")
df5 = pd.read_csv(DATA_DIR / "案例5_人力资源明细.csv")
df6 = pd.read_csv(DATA_DIR / "案例6_玉玦图流量来源.csv")
df7 = pd.read_csv(DATA_DIR / "案例7_区域月度完成率.csv")
print("数据加载完成:", {k: v.shape for k, v in
      {"df1": df1, "df2": df2, "df3": df3, "df4": df4, "df5": df5, "df6": df6, "df7": df7}.items()})

## 案例 1：动态柱形图（各区域月度销售额）

**Excel 原理**：控件单元格（J3）选择月份 → `INDEX(B4:B9, J3)` 取出月份、`VLOOKUP` 整行取数 → 柱形图随之刷新。

**Python 实现**：`Timeline` 时间轴替代控件单元格，点击/拖动即可切换 1—6 月，等价于 INDEX+VLOOKUP 的动态取数效果。

In [ ]:
# ============================================================
# 案例 1：动态柱形图 —— Bar + Timeline
# ============================================================
regions = df1.columns[1:].tolist()

tl1 = Timeline(init_opts=opts.InitOpts(width="920px", height="520px"))
for _, row in df1.iterrows():
    bar = (
        Bar()
        .add_xaxis(regions)
        .add_yaxis(
            "销售额", row[regions].tolist(),
            label_opts=opts.LabelOpts(position="top", font_size=13),
            itemstyle_opts=opts.ItemStyleOpts(color=JsCode(
                "function(p){var cs=" + str(PALETTE) + ";return cs[p.dataIndex % cs.length];}")),
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title=f"{row['月份']}各区域销售额", subtitle="拖动下方时间轴切换月份"),
            yaxis_opts=opts.AxisOpts(name="销售额", max_=1000),
            legend_opts=opts.LegendOpts(is_show=False),
        )
    )
    tl1.add(bar, row["月份"])

tl1.add_schema(play_interval=1500, is_auto_play=False, is_loop_play=False,
               pos_bottom="0%", control_position="left")
tl1.render(str(OUT_DIR / "案例1_动态柱形图.html"))
tl1.render_notebook()

## 案例 2：动态跑道图（各部门月度人数）

**Excel 原理**：每个部门一个圆环系列，系列数据为 `[部门人数, 占位]`，其中 **占位 = 当月总人数 ÷ 2 − 部门人数**，使每个圆环形成"跑道"式进度条；圆环中心用公式 `月份&"公司总人数为"&SUM(...)` 动态显示合计。

**Python 实现**：`Pie` 多环嵌套（每部门一环、占位段置灰），`Timeline` 切换月份，中心标题动态显示总人数。

In [ ]:
# ============================================================
# 案例 2：动态跑道图 —— 多环嵌套 Pie + Timeline
# ============================================================
months2 = ["1月", "2月", "3月", "4月"]
depts = df2["部门"].tolist()

# 只给"数值段"显示标签，占位段隐藏
label_only_value = JsCode(
    "function(p){return p.name === '占位' ? '' : p.name + '\n' + p.value + '人';}")

tl2 = Timeline(init_opts=opts.InitOpts(width="920px", height="600px"))
for m in months2:
    total = int(df2[m].sum())                      # 当月公司总人数
    pie = Pie(init_opts=opts.InitOpts(width="920px", height="600px"))
    for i, dept in enumerate(depts):
        v = int(df2.loc[df2["部门"] == dept, m].iloc[0])
        placeholder = total / 2 - v                # 占位 = 总人数/2 − 部门人数（与 Excel 一致）
        inner, outer = 18 + i * 12, 18 + i * 12 + 10   # 每环宽度 10%、间隔 2%
        pie.add(
            dept,
            [(dept, v), ("占位", round(placeholder, 1))],
            radius=[f"{inner}%", f"{outer}%"],
            label_opts=opts.LabelOpts(formatter=label_only_value, font_size=11),
        )
    pie.set_colors(PALETTE + ["#EBEBEB"] * 6)      # 占位段浅灰
    pie.set_global_opts(
        title_opts=opts.TitleOpts(
            title=f"{m}\n公司总人数为 {total}",
            pos_left="center", pos_top="middle",
            title_textstyle_opts=opts.TextStyleOpts(font_size=18, color="#333")),
        legend_opts=opts.LegendOpts(is_show=False),
        tooltip_opts=opts.TooltipOpts(formatter=JsCode(
            "function(p){return p.name === '占位' ? '' : p.seriesName + '：' + p.value + '人';}")),
    )
    tl2.add(pie, m)

tl2.add_schema(play_interval=1800, is_auto_play=False, is_loop_play=False, pos_bottom="0%")
tl2.render(str(OUT_DIR / "案例2_动态跑道图.html"))
tl2.render_notebook()

## 案例 3：动态南丁格尔圆环图（流量来源占比）

**Excel 原理**：控件输入单元格（I3）选择周期（1=近7天 / 2=近30天）→ 图表数据行随之切换 → 南丁格尔玫瑰图刷新。

**Python 实现**：`Pie(rosetype="radius")` 绘制南丁格尔玫瑰图（扇区角度相同、半径表示数值），`Timeline` 切换统计周期。

In [ ]:
# ============================================================
# 案例 3：动态南丁格尔圆环图 —— Pie(rosetype) + Timeline
# ============================================================
sources3 = df3.columns[1:].tolist()

tl3 = Timeline(init_opts=opts.InitOpts(width="920px", height="560px"))
for _, row in df3.iterrows():
    period = row["周期"]
    data = [(s, round(float(row[s]), 4)) for s in sources3]
    rose = (
        Pie(init_opts=opts.InitOpts(width="920px", height="560px"))
        .add(
            period, data,
            radius=["20%", "70%"],            # 内半径形成圆环
            center=["50%", "52%"],
            rosetype="radius",                # 南丁格尔玫瑰图：半径映射数值
            label_opts=opts.LabelOpts(formatter="{b}\n{d}%", font_size=13),
        )
        .set_colors(PALETTE)
        .set_global_opts(
            title_opts=opts.TitleOpts(title=f"流量来源占比（{period}）"),
            legend_opts=opts.LegendOpts(orient="vertical", pos_left="right", pos_top="middle"),
            tooltip_opts=opts.TooltipOpts(formatter="{b}: {c} ({d}%)"),
        )
    )
    tl3.add(rose, period)

tl3.add_schema(play_interval=2000, is_auto_play=False, is_loop_play=False, pos_bottom="0%")
tl3.render(str(OUT_DIR / "案例3_动态南丁格尔圆环图.html"))
tl3.render_notebook()

## 案例 4：动态组合图（2022 年化妆品销售情况）

**Excel 原理**：3 个复选框分别控制"销售额 / 利润 / 利润率"三个系列，公式 `IF($C$13, F4, -100)` 把取消勾选的系列压到坐标轴之外实现"隐藏"；柱形（销售额、利润）+ 折线（利润率，次坐标轴）组合。

**Python 实现**：`Bar + Line` 双轴组合图，用**图例点击**替代复选框控制系列显隐（点击图例中的"销售额/利润/利润率"即可开关对应系列，试试！）。

In [ ]:
# ============================================================
# 案例 4：动态组合图 —— Bar + Line 双轴，图例控制显隐
# ============================================================
cats4 = df4["类别"].tolist()

bar4 = (
    Bar(init_opts=opts.InitOpts(width="920px", height="540px"))
    .add_xaxis(cats4)
    .add_yaxis("销售额", df4["销售额"].tolist(), color="#4472C4",
               label_opts=opts.LabelOpts(position="top"))
    .add_yaxis("利润", df4["利润"].tolist(), color="#ED7D31",
               label_opts=opts.LabelOpts(position="top"))
    .extend_axis(
        yaxis=opts.AxisOpts(
            name="利润率", min_=0, max_=1,
            axislabel_opts=opts.LabelOpts(formatter=JsCode(
                "function(v){return (v * 100).toFixed(0) + '%';}")),
            splitline_opts=opts.SplitLineOpts(is_show=False),
        )
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="2022年化妆品销售情况",
                                  subtitle="点击图例可显示/隐藏对应系列（等价于 Excel 复选框）"),
        yaxis_opts=opts.AxisOpts(name="金额（元）"),
        legend_opts=opts.LegendOpts(pos_top="12%"),
        tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="shadow"),
    )
)
line4 = (
    Line()
    .add_xaxis(cats4)
    .add_yaxis("利润率", df4["利润率"].tolist(), yaxis_index=1,
               color="#70AD47", symbol_size=12, is_smooth=True,
               label_opts=opts.LabelOpts(formatter=JsCode(
                   "function(p){var v = Array.isArray(p.value) ? p.value[1] : p.value;"
                   "return (v * 100).toFixed(1) + '%';}"),
                   position="right", distance=10, color="#70AD47", font_weight="bold"))
)
combo4 = bar4.overlap(line4)
combo4.render(str(OUT_DIR / "案例4_动态组合图.html"))
combo4.render_notebook()

## 案例 5：数据透视表 + 切片器（人力资源数据分析）

**Excel 原理**：1470 条员工明细 → 数据透视表按"学历"统计平均月收入 → **切片器**按部门筛选（文件中的切片器停留在「市场拓展」部门）。

**Python 实现**：
- `pandas.pivot_table` 等价于数据透视表；
- pyecharts `Tab` 选项卡等价于切片器，点击部门即可切换该部门下各学历的平均月收入。

先用 pandas 复现透视表（含切片器「市场拓展」筛选状态，与 Excel 数值完全一致）：

In [ ]:
# ============================================================
# 案例 5-1：pandas 复现数据透视表（切片器状态：市场拓展）
# ============================================================
edu_order = ["高中", "专科", "本科", "硕士", "博士"]

pivot_all = (df5.pivot_table(index="学历", values="月收入", aggfunc="mean")
             .reindex(edu_order).round(2))
pivot_mkt = (df5[df5["部门"] == "市场拓展"]
             .pivot_table(index="学历", values="月收入", aggfunc="mean")
             .reindex(edu_order).round(2))

pivot_show = pd.DataFrame({
    "平均月收入(全部员工)": pivot_all["月收入"],
    "平均月收入(切片器:市场拓展)": pivot_mkt["月收入"],
})
pivot_show.loc["总计"] = [round(df5["月收入"].mean(), 2),
                         round(df5[df5["部门"] == "市场拓展"]["月收入"].mean(), 2)]
print(f"员工总数: {len(df5)} 人 | 部门数: {df5['部门'].nunique()} 个")
pivot_show

In [ ]:
# ============================================================
# 案例 5-2：Tab 选项卡 = 切片器 —— 各部门各学历平均月收入
# ============================================================
tab5 = Tab()
dept_list = ["全部"] + sorted(df5["部门"].unique().tolist())

for dept in dept_list:
    sub = df5 if dept == "全部" else df5[df5["部门"] == dept]
    g = sub.groupby("学历")["月收入"].mean().reindex(edu_order).round(0)
    bar = (
        Bar(init_opts=opts.InitOpts(width="920px", height="500px"))
        .add_xaxis(edu_order)
        .add_yaxis("平均月收入", g.tolist(), color="#4472C4",
                   label_opts=opts.LabelOpts(position="top", formatter="{c} 元"))
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title=f"各学历平均月收入（{dept}，{len(sub)} 人）"),
            yaxis_opts=opts.AxisOpts(name="平均月收入（元）"),
            legend_opts=opts.LegendOpts(is_show=False),
        )
    )
    tab5.add(bar, dept)

tab5.render(str(OUT_DIR / "案例5_透视表切片器.html"))
tab5.render_notebook()

## 案例 6：VBA 动态玉玦图（流量来源）

**Excel 原理**：两个按钮分别绑定 VBA 宏 `近7天()` / `近30天()`，点击按钮 → 宏把周期写入控件单元格 H3 → 图表数据区按周期取数，每个来源一个圆环系列 `[占比, 占位(0.5−占比)]`，形成带缺口的"玉玦"造型。

```vba
Sub 近7天()
    Range("H3").Select
    ActiveCell.FormulaR1C1 = "近7天"
End Sub
```

**Python 实现**：无需宏——`Tab` 选项卡就是"两个按钮"，每来源一环、占位段置灰，点击选项卡即切换周期。

In [ ]:
# ============================================================
# 案例 6：玉玦图 —— 多环 Pie(数值+占位) + Tab(替代 VBA 按钮)
# ============================================================
sources6 = df6.columns[1:].tolist()   # 个人主页/搜索/关注页面/首页推荐
label_pct = JsCode(
    "function(p){return p.name === '占位' ? '' : p.name + '\n' + (p.value * 100).toFixed(0) + '%';}")

tab6 = Tab()
for _, row in df6.iterrows():
    period = row["周期"]
    pie = Pie(init_opts=opts.InitOpts(width="920px", height="560px"))
    for i, s in enumerate(sources6):
        v = round(float(row[s]), 4)
        inner, outer = 15 + i * 14, 15 + i * 14 + 11
        pie.add(
            s,
            [(s, v), ("占位", round(0.5 - v, 4))],   # 占位 = 0.5 − 占比（与 Excel 一致）
            radius=[f"{inner}%", f"{outer}%"],
            label_opts=opts.LabelOpts(formatter=label_pct, font_size=12),
        )
    pie.set_colors(["#ED7D31", "#A5A5A5", "#FFC000", "#4472C4"] + ["#EBEBEB"] * 4)
    pie.set_global_opts(
        title_opts=opts.TitleOpts(title=f"流量来源玉玦图（{period}）", pos_top="2%"),
        legend_opts=opts.LegendOpts(is_show=False),
        tooltip_opts=opts.TooltipOpts(formatter=JsCode(
            "function(p){return p.name === '占位' ? '' : p.seriesName + '：' + (p.value * 100).toFixed(0) + '%';}")),
    )
    tab6.add(pie, period)   # 两个选项卡 ≈ 两个 VBA 按钮

tab6.render(str(OUT_DIR / "案例6_VBA动态玉玦图.html"))
tab6.render_notebook()

## 案例 7：动态滑珠图（各区域月度目标完成率）

**Excel 原理**：组合框/选项按钮选择月份 → 图表数据区计算 `完成率`、`未完成率(=1−完成率)` 与 `辅助` 定位列 → 堆积柱形图（完成率+透明的未完成率=100% 滑轨）+ 散点（滑珠定位在完成率顶端）。

**Python 实现**：`Bar` 堆积（完成率 + 未完成率浅色滑轨）叠加 `EffectScatter` 涟漪滑珠，`Timeline` 切换月份。

In [ ]:
# ============================================================
# 案例 7：动态滑珠图 —— 堆积 Bar + EffectScatter + Timeline
# ============================================================
regions7 = df7["区域"].tolist()
months7 = df7.columns[1:].tolist()

tl7 = Timeline(init_opts=opts.InitOpts(width="920px", height="540px"))
for m in months7:
    rates = [round(float(v), 4) for v in df7[m].tolist()]
    bar = (
        Bar(init_opts=opts.InitOpts(width="920px", height="540px"))
        .add_xaxis(regions7)
        .add_yaxis("完成率", rates, stack="slider", color="#4472C4", bar_width="45%",
                   label_opts=opts.LabelOpts(is_show=False))
        .add_yaxis("未完成率", [round(1 - r, 4) for r in rates], stack="slider",
                   color="#DCE6F1", bar_width="45%",     # 浅色滑轨
                   label_opts=opts.LabelOpts(is_show=False))
        .set_global_opts(
            title_opts=opts.TitleOpts(title=f"{m}各区域目标完成率", subtitle="滑珠位置即完成率水平"),
            legend_opts=opts.LegendOpts(is_show=False),
            xaxis_opts=opts.AxisOpts(name="区域"),
            yaxis_opts=opts.AxisOpts(name="完成率", max_=1,
                                     axislabel_opts=opts.LabelOpts(formatter=JsCode(
                                         "function(v){return (v*100).toFixed(0)+'%';}"))),
            tooltip_opts=opts.TooltipOpts(trigger="axis", formatter=JsCode(
                "function(ps){var p=ps[0];var v=Array.isArray(p.value)?p.value[1]:p.value;"
                "return p.name+'：完成率 '+(v*100).toFixed(1)+'%';}")),
        )
    )
    bead = (
        EffectScatter()
        .add_xaxis(regions7)
        .add_yaxis("滑珠", rates, symbol_size=16, color="#ED7D31",
                   label_opts=opts.LabelOpts(is_show=False))
    )
    bar.overlap(bead)
    tl7.add(bar, m)

tl7.add_schema(play_interval=1500, is_auto_play=False, is_loop_play=False, pos_bottom="0%")
tl7.render(str(OUT_DIR / "案例7_动态滑珠图.html"))
tl7.render_notebook()

## 综合应用：数据大屏（从图表到数据大屏）

呼应教材书名《Excel数据可视化——从图表到数据大屏》，用 pyecharts `Page` 把 7 个案例整合为一张**可拖拽布局的数据大屏**（保存为 HTML 后可在浏览器中自由拖动每个图表的位置与大小）。

In [ ]:
# ============================================================
# 数据大屏：Page 可拖拽布局，整合全部案例
# ============================================================
# 注：pyecharts 2.x 的 Page 不能嵌套 Tab 组件，
# 因此大屏中案例5/案例6 各取一张代表性图表（全部部门、近30天）
g5 = df5.groupby("学历")["月收入"].mean().reindex(edu_order).round(0)
bar5_all = (
    Bar(init_opts=opts.InitOpts(width="920px", height="500px"))
    .add_xaxis(edu_order)
    .add_yaxis("平均月收入", g5.tolist(), color="#4472C4",
               label_opts=opts.LabelOpts(position="top", formatter="{c} 元"))
    .set_global_opts(
        title_opts=opts.TitleOpts(title=f"各学历平均月收入（全部，{len(df5)} 人）"),
        yaxis_opts=opts.AxisOpts(name="平均月收入（元）"),
        legend_opts=opts.LegendOpts(is_show=False))
)
row6 = df6[df6["周期"] == "近30天"].iloc[0]
pie6_30 = Pie(init_opts=opts.InitOpts(width="920px", height="560px"))
for i, s in enumerate(sources6):
    v = round(float(row6[s]), 4)
    inner, outer = 15 + i * 14, 15 + i * 14 + 11
    pie6_30.add(s, [(s, v), ("占位", round(0.5 - v, 4))],
                radius=[f"{inner}%", f"{outer}%"],
                label_opts=opts.LabelOpts(formatter=label_pct, font_size=12))
pie6_30.set_colors(["#ED7D31", "#A5A5A5", "#FFC000", "#4472C4"] + ["#EBEBEB"] * 4)
pie6_30.set_global_opts(
    title_opts=opts.TitleOpts(title="流量来源玉玦图（近30天）", pos_top="2%"),
    legend_opts=opts.LegendOpts(is_show=False))

page = Page(page_title="第三章 动态图表 · Python 数据大屏",
            layout=Page.DraggablePageLayout)
page.add(tl1, tl2, tl3, combo4, bar5_all, pie6_30, tl7)
page.render(str(OUT_DIR / "数据大屏.html"))
print("数据大屏已生成: output/数据大屏.html （浏览器打开后可自由拖拽布局）")
page.render_notebook()

---

## 实验总结

| 案例 | Excel 动态机制 | Python 实现 | 输出文件 |
|---|---|---|---|
| 1 动态柱形图 | 控件单元格 + INDEX/VLOOKUP | `Bar + Timeline` | 案例1_动态柱形图.html |
| 2 动态跑道图 | 圆环图 + 占位（总人数/2−数值） | 多环 `Pie + Timeline` | 案例2_动态跑道图.html |
| 3 动态南丁格尔圆环图 | 控件输入切换周期 | `Pie(rosetype) + Timeline` | 案例3_动态南丁格尔圆环图.html |
| 4 动态组合图 | 复选框 + IF 控制显隐 | `Bar + Line` 双轴 + 图例开关 | 案例4_动态组合图.html |
| 5 透视表切片器 | 数据透视表 + 切片器 | `pivot_table + Tab` | 案例5_透视表切片器.html |
| 6 VBA动态玉玦图 | VBA 宏按钮 | 多环 `Pie + Tab` | 案例6_VBA动态玉玦图.html |
| 7 动态滑珠图 | 组合框 + 选项按钮 + 辅助列 | 堆积 `Bar + EffectScatter + Timeline` | 案例7_动态滑珠图.html |
| 综合 | —— | `Page` 可拖拽大屏 | 数据大屏.html |

**迁移体会**：
1. Excel 的"控件 → 公式取数 → 图表刷新"链路，在 pyecharts 中由 `Timeline / Tab / Legend` 等交互组件原生支持，无需控件与宏；
2. Excel 依赖 `INDEX/VLOOKUP/IF` 构造"图表数据区"，Python 中由 pandas 直接完成数据筛选与聚合，逻辑更直观；
3. VBA 宏按钮的功能被 `Tab` 选项卡完全替代，摆脱了对宏安全性的依赖，且生成的 HTML 可直接部署为 Web 数据大屏。